# 02 - PPO 数据流与 GAE

本节目标: 你能手算 `returns` 和 `advantages`, 并知道为什么 PPO 要倒序算 GAE。

In [ ]:
import torch
torch.manual_seed(0)

## PPO rollout 里存什么

一次 step 会存:

```text
obs, action, reward, done, value, old_log_prob, old_distribution_params
```

`reward` 来自环境, `value` 来自 critic, `old_log_prob` 来自 rollout 时的 actor。

## 手写 GAE

公式:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

$$A_t = \delta_t + \gamma \lambda A_{t+1}$$

$$R_t = A_t + V(s_t)$$

In [ ]:
rewards = torch.tensor([1.0, 2.0, 3.0])
values = torch.tensor([0.5, 1.0, 1.5])
last_value = torch.tensor(2.0)
dones = torch.tensor([0.0, 0.0, 0.0])
gamma = 0.99
lam = 0.95

advantage = torch.tensor(0.0)
advantages = torch.zeros(3)
returns = torch.zeros(3)

for step in reversed(range(3)):
    next_value = last_value if step == 2 else values[step + 1]
    not_done = 1.0 - dones[step]
    delta = rewards[step] + not_done * gamma * next_value - values[step]
    advantage = delta + not_done * gamma * lam * advantage
    advantages[step] = advantage
    returns[step] = advantage + values[step]
    print(step, 'delta=', round(delta.item(), 4), 'adv=', round(advantage.item(), 4))

print('advantages:', advantages)
print('returns:', returns)

## done 会切断 bootstrap

如果某一步 `done=True`, 后面就不是同一个 episode, 不能从下一个 value bootstrap。

In [ ]:
dones = torch.tensor([1.0, 0.0, 0.0])
advantage = torch.tensor(0.0)
advantages_done = torch.zeros(3)

for step in reversed(range(3)):
    next_value = last_value if step == 2 else values[step + 1]
    not_done = 1.0 - dones[step]
    delta = rewards[step] + not_done * gamma * next_value - values[step]
    advantage = delta + not_done * gamma * lam * advantage
    advantages_done[step] = advantage

print('advantages with done at step 0:', advantages_done)

## 作业

1. 把 `dones = [0, 1, 0]`, 预测 step 0 的 advantage 会不会受 step 2 影响。
2. 把 `lambda` 改成 0, 解释此时 GAE 退化成什么。
3. 把 `gamma` 改成 0, 解释 return 会发生什么。